# E2: which camera colour model predicts held-out frames? (Colab, T4 is enough)

**No training.** For each trained uwsplat model (`gs/mvs_lpips/ckpt.pt` on Drive), the raw render of every training frame is mapped to its photo by six per-frame colour models, refitted by least squares. A held-out frame takes its parameters from its two temporal training neighbours, interpolated (gains in log space). About 10 minutes for setup, then about a minute per scene.

| family | per-frame model | parameters per frame |
|---|---|---|
| `A` | I = W_k R + b_k (the form of uwsplat's Appearance module) | 12 |
| `B` | I = W_k R | 9 |
| `C` | I = M diag(g_k) R, shared M | 3 |
| `D` | I = diag(g_k) R | 3 |
| `Do` | I = diag(g_k) R + o_k | 6 |
| `E` | I = e_k R | 1 |

Reference rows: `identity` (no correction) and `trained` (the model's own Appearance correction, interpolated the same way = our 24.76 dB `neighbors` protocol on Submerged3D).

**Why.** `scripts/identifiability_check.py` (cell 4) shows on synthetic data, with every null-space dimension as predicted:
- per-frame offsets are confounded with backscatter: exactly when attenuation and backscatter coefficients are equal, and nearly so otherwise (about 3,500× weaker than any other direction);
- a shared colour matrix is pinned to diagonal by white balance that changes from frame to frame, in air as much as underwater.

So the offsets in `A` should absorb water residuals that do not interpolate in time, and the gauge-minimal `C` / `D` should predict held-out frames better.

**Pre-registered kill rule** (from *Novel contributions beyond RUSplatting*, E2): **C or D must beat A on test PSNR by more than 0.05 dB in ≥ 3 of 4 scenes**, otherwise the camera track is killed. (The 0.05 dB margin was added before any real run, so that ties don't count as wins.) Secondary, not part of the rule: does the best family beat `trained`?

**Caveat.** The frozen model was trained with its own per-frame affine correction, so this is a post-hoc proxy. A positive result would justify retraining with the constrained camera, not replace it. Submerged3D is JPEG (not linear), and SeaThru-NeRF is photos, not video (its "temporal" neighbours are file-name neighbours).

In [ ]:
# ---- 1. settings ----
DRIVE_DIR = "/content/drive/MyDrive/uwsplat_colab"
SETS = {"Submerged3D": f"{DRIVE_DIR}/outputs_v2", "SeaThru-NeRF": f"{DRIVE_DIR}/outputs_seathru"}
SCENES = None                 # None = every scene with gs/<TAG>/ckpt.pt
TAG = "mvs_lpips"
FAMILIES = "A,B,C,D,Do,E"
STRIDE = 2                    # fit on every 2nd pixel in each direction
FORCE = False
RES_DRIVE = f"{DRIVE_DIR}/camera_families"
REPO, WORK = "/content/splat_dense-sparse", "/content/cam_scenes"

import os, subprocess, torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU (T4 is fine)"
cc = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"
print("torch", torch.__version__, "CUDA", torch.version.cuda)

In [ ]:
# ---- 2. Drive ----
from google.colab import drive
drive.mount("/content/drive", timeout_ms=90000)
for dset, root in SETS.items():
    print(f"{dset:13s} {root}:", sorted(os.listdir(root)) if os.path.isdir(root) else "NOT FOUND")
os.makedirs(RES_DRIVE, exist_ok=True)

In [ ]:
%%bash -s "$REPO"
# ---- 3. code (branch dense-v2) and packages ----
set -e
if [ -d "$1/.git" ]; then git -C "$1" fetch -q origin dense-v2 && git -C "$1" reset -q --hard origin/dense-v2
else git clone -q --branch dense-v2 https://github.com/noUpKey/splat_dense-sparse "$1"; fi
git -C "$1" log --oneline -1
test -f "$1/uwsplat/eval/camera_families.py" || { echo "camera_families.py missing: wrong branch?"; exit 1; }
pip -q install "lpips==0.1.4" safetensors ninja "gsplat==1.5.3"

In [ ]:
# ---- 3b. compile gsplat once (5-10 min on a fresh runtime; skip if the C2 notebook already ran in this runtime) ----
import time
t0 = time.time()
from gsplat import rasterization
dev = "cuda"
img, alpha, _ = rasterization(torch.tensor([[0.0, 0.0, 3.0]], device=dev), torch.tensor([[1.0, 0, 0, 0]], device=dev),
                              torch.full((1, 3), 0.1, device=dev), torch.tensor([0.9], device=dev),
                              torch.ones(1, 1, 3, device=dev), torch.eye(4, device=dev)[None],
                              torch.tensor([[[50.0, 0, 32], [0, 50.0, 32], [0, 0, 1]]], device=dev), 64, 64,
                              render_mode="RGB+ED")
print(f"gsplat OK ({time.time() - t0:.0f} s)")

In [ ]:
# ---- 4. CPU checks: the identifiability maths (null spaces as predicted) and the camera-family fits ----
for cmd in (["python", "scripts/identifiability_check.py"], ["python", "-m", "tests.test_camera_families"]):
    r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn" not in l)[-2500:])
    assert r.returncode == 0, f"{cmd} failed"

In [ ]:
# ---- 5. scenes on Drive ----
import glob
jobs = []
for dset, root in SETS.items():
    if not os.path.isdir(root):
        continue
    for sd in sorted(glob.glob(f"{root}/*/")):
        sd = sd.rstrip("/")
        scene = os.path.basename(sd)
        if (SCENES and scene not in SCENES) or not os.path.exists(f"{sd}/sfm/scene.json"):
            continue
        has = os.path.exists(f"{sd}/gs/{TAG}/ckpt.pt")
        done = os.path.exists(f"{RES_DRIVE}/{dset}/{scene}/camera_families.json")
        jobs.append(dict(dset=dset, scene=scene, src=sd, has_ckpt=has, done=done))
        print(f"{dset:13s} {scene:24s} checkpoint: {'yes' if has else 'MISSING'}"
              f"{'   (result on Drive: skip)' if done and not FORCE else ''}")

In [ ]:
# ---- 6. run ----
import json, shutil, sys

def sh(cmd):
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(cmd)}")

failed = []
for j in jobs:
    if not j["has_ckpt"] or (j["done"] and not FORCE):
        continue
    local = f"{WORK}/{j['dset']}/{j['scene']}"
    dst = f"{RES_DRIVE}/{j['dset']}/{j['scene']}"
    print(f"\n===== {j['dset']} / {j['scene']}")
    try:
        for sub in ("sfm", "images"):
            shutil.copytree(f"{j['src']}/{sub}", f"{local}/{sub}", dirs_exist_ok=True)
        os.makedirs(f"{local}/gs/{TAG}", exist_ok=True)
        shutil.copy2(f"{j['src']}/gs/{TAG}/ckpt.pt", f"{local}/gs/{TAG}/ckpt.pt")
        sh([sys.executable, "scripts/camera_families.py", "--scene_out", local, "--tag", TAG,
            "--families", FAMILIES, "--stride", str(STRIDE)])
        os.makedirs(dst, exist_ok=True)
        shutil.copy2(f"{local}/eval/{TAG}/camera_families.json", f"{dst}/camera_families.json")
        shutil.rmtree(local, ignore_errors=True)
    except Exception as e:
        print("ERROR:", e)
        failed.append(f"{j['dset']}/{j['scene']}")
print("\nfailed:", failed or "none")

## 7. Results and the kill rule
Test = held-out views with neighbour-interpolated parameters. Train LOO = each training frame predicted from its neighbours without itself. Own = each training frame with its own fit (how much the family can absorb; high own with low LOO = overfitting).

In [ ]:
import math
import numpy as np

fams = FAMILIES.split(",")
keys = ["identity", "trained"] + fams
res = {}
for f in sorted(glob.glob(f"{RES_DRIVE}/*/*/camera_families.json")):
    res.setdefault(f.split("/")[-3], {})[f.split("/")[-2]] = json.load(open(f))
lines = [f"# E2 camera families ({TAG}, stride {STRIDE})", ""]
def out(s=""):
    print(s); lines.append(s)

for dset, scenes in res.items():
    out(f"## {dset}")
    out("")
    out("Test PSNR (neighbour-interpolated parameters):")
    out("")
    out("| scene | " + " | ".join(keys) + " |")
    out("|---|" + "---|" * len(keys))
    for sc, r in scenes.items():
        out(f"| {sc} | " + " | ".join(f"{r['summary']['test'][k]['psnr']:.2f}" for k in keys) + " |")
    m = {k: np.mean([r["summary"]["test"][k]["psnr"] for r in scenes.values()]) for k in keys}
    out("| **mean** | " + " | ".join(f"**{m[k]:.2f}**" for k in keys) + " |")
    out("")
    for metric in ("ssim", "lpips"):
        mm = {k: np.mean([r["summary"]["test"][k][metric] for r in scenes.values()]) for k in keys}
        out(f"- mean test {metric.upper()}: " + ", ".join(f"{k} {mm[k]:.4f}" for k in keys))
    lo = {k: np.mean([r["summary"]["train_loo_psnr"][k] for r in scenes.values()]) for k in keys}
    ow = {k: np.mean([r["summary"]["train_own_psnr"][k] for r in scenes.values()]) for k in fams}
    out("- mean train leave-one-out PSNR: " + ", ".join(f"{k} {lo[k]:.2f}" for k in keys))
    out("- mean train own-fit PSNR: " + ", ".join(f"{k} {ow[k]:.2f}" for k in fams))
    out("")
    # kill rule
    n = len(scenes)
    need = max(1, math.ceil(0.75 * n))
    wins, margins = 0, []
    for sc, r in scenes.items():
        t = r["summary"]["test"]
        best = max(t[k]["psnr"] for k in ("C", "D") if k in t)
        margins.append(f"{sc} {best - t['A']['psnr']:+.2f}")
        wins += best > t["A"]["psnr"] + 0.05
    out(f"**Kill rule, {dset}:** C or D beats A on test PSNR by > 0.05 dB in ≥ {need}/{n} scenes: {wins}/{n} "
        f"({', '.join(margins)}) -> **{'CAMERA TRACK SURVIVES' if wins >= need else 'KILLED (as pre-registered)'}**")
    best_fam = max(fams, key=lambda k: m[k])
    out(f"- secondary: best family {best_fam} {m[best_fam]:.2f} vs trained {m['trained']:.2f} dB ({m[best_fam] - m['trained']:+.2f})")
    out("")

with open(f"{RES_DRIVE}/summary.md", "w") as fh:
    fh.write("\n".join(lines) + "\n")
print(f"saved {RES_DRIVE}/summary.md")